<a href="https://colab.research.google.com/github/UmerSajid842/Fraud-detection-system/blob/main/GrokPaysimcode.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Spatio-Temporal Fraud Detection with Adaptive Transformers, Graph Proposal Neural Networks, and LLM-Based Trustworthy Explanations
PaySim Dataset Edition
Research Title: Spatio Temporal Fraud Detection with Adaptive Transformers, Graph Proposal Neural Networks, and LLM Based Trustworthy Explanations

Dataset: PaySim (Synthetic Mobile Money Transactions)

Primary Metrics: PR-AUC (most important), ROC-AUC, Precision, Recall, F1, MCC

In [1]:
Property,Value,Implication for Model Design
Rows,~6.36 million (often subsampled),Need scalable graph / mini-batch training
Fraud Rate,~0.13%,Extreme imbalance → Focal Loss + careful sampling
Time,"step (1 step = 1 hour, 744 steps)",Strong temporal signal
Entities,"nameOrig, nameDest",Natural multi-relational graph (accounts)
Transaction Types,"CASH-IN, CASH-OUT, DEBIT, PAYMENT, TRANSFER",Heterogeneous edges / type-aware attention
Amount & Balances,Continuous features,Useful for edge attributes and node features

SyntaxError: invalid character '→' (U+2192) (2288289084.py, line 3)

Key Advantage over European Credit Card:

PaySim has explicit account identifiers (nameOrig / nameDest). This allows a true Graph Proposal Neural Network that builds an account-transaction graph, which is central to your research title.

#,Model,Why Suitable for PaySim,Novelty for Your Title
1,GraphSAGE / GAT,Classic message passing on account graph,Medium
2,Temporal Graph Network (TGN),Continuous-time edges using step,High
3,Heterogeneous Graph Transformer,"Different edge types (TRANSFER, CASH_OUT, etc.)",High
4,SpaTempFraudNet-PaySim (Recommended),Adaptive Transformer + Graph Proposal (account graph) + Temporal Encoding + LLM explanations,Highest

Recommendation: Design a Customized Model
Existing models (GraphSAGE, TGN, TabTransformer) are strong baselines, but none jointly optimize:

Adaptive feature interactions (Transformer)
Explicit account-level relational structure (Graph Proposal)
Temporal dynamics (step)
Trustworthy natural-language explanations (LLM)

→ SpaTempFraudNet-PaySim is the best match for your title.

3. Technical & Mathematical Justification
Problem 1 – Extreme Class Imbalance (~0.13%)
Focal Loss focuses learning on hard minority examples:
$$L_{\text{focal}} = -\alpha_t (1 - p_t)^\gamma \log(p_t)$$
Recommended settings for PaySim:
$\gamma = 2$, $\alpha \approx 0.999$ (close to $1 - \text{fraud rate}$).
Problem 2 – Relational Fraud Patterns (Account Takeover / Money Mules)
Fraudsters often use the same accounts in sequences (TRANSFER → CASH_OUT).
Graph Neural Networks capture multi-hop patterns via message passing:
$$\mathbf{h}_v^{(l+1)} = \sigma \left( \mathbf{W} \cdot \text{AGG} \left( \{\mathbf{h}_u^{(l)} : u \in \mathcal{N}(v)\} \right) \right)$$
or multi-head Graph Attention:
$$\alpha_{ij} = \text{softmax}_j \left( \frac{(\mathbf{W} \mathbf{h}_i)^\top (\mathbf{W} \mathbf{h}_j)}{\sqrt{d}} \right)$$
Problem 3 – Temporal Dynamics
step provides a natural time axis. Temporal encoding or continuous-time kernels:
$$\mathbf{h}_i = \mathbf{h}_i + \text{TE}(\Delta t_i) \quad \text{or} \quad \alpha_{ij} \propto \exp\left( -\frac{|\text{step}_i - \text{step}_j|}{\tau} \right)$$
Problem 4 – Heterogeneous Transaction Types
Different edge types (TRANSFER vs CASH_OUT) carry different semantics. Type-aware attention or relational GAT improves discrimination.
Problem 5 – Trustworthy Explanations
Attention weights from GAT + Transformer + local subgraph of involved accounts are fed to an LLM to generate human-readable explanations. SHAP values on tabular features can be added for completeness.


4. Complete Modular Pipeline

In [ ]:
Raw PaySim CSV
  → Module 1: Load & Basic Cleaning
  → Module 2: Feature Engineering & Encoding
  → Module 3: Graph Proposal (Account Graph)
  → Module 4: Model Definition (SpaTempFraudNet-PaySim)
  → Module 5: Training with Focal Loss
  → Module 6: Full Evaluation (PR-AUC, ROC-AUC, Precision, Recall, F1, MCC)
  → Module 7: LLM-based Trustworthy Explanations

Module 1: Data Loading & Basic Cleaning
Purpose: Load PaySim, handle missing values, basic filtering, and save a clean version.

Important: Many papers avoid using post-transaction balances for fraud detection because fraudulent transactions are cancelled in the simulator. We keep them for feature engineering but document the limitation.

In [8]:
# ============================================================
# MODULE 1: DATA LOADING & BASIC CLEANING
# Purpose: Load PaySim CSV, remove duplicates, basic sanity checks,
#          and persist cleaned raw data.
# ============================================================

import pandas as pd
import numpy as np
import os

os.makedirs("artifacts", exist_ok=True)

def load_paysim(path: str = "PS_20174392719_1491204439457_log.csv") -> pd.DataFrame:
    """
    Load the PaySim dataset.
    Expected columns:
    step, type, amount, nameOrig, oldbalanceOrg, newbalanceOrig,
    nameDest, oldbalanceDest, newbalanceDest, isFraud, isFlaggedFraud
    """
    df = pd.read_csv(path)

    # Basic cleaning
    df = df.drop_duplicates().reset_index(drop=True)

    # Ensure correct types
    df["isFraud"] = df["isFraud"].astype(int)
    df["isFlaggedFraud"] = df["isFlaggedFraud"].astype(int)

    print(f"Loaded shape     : {df.shape}")
    print(f"Fraud rate       : {df['isFraud'].mean():.6f}")
    print(f"Fraud count      : {df['isFraud'].sum()}")
    print(f"Transaction types: {df['type'].value_counts().to_dict()}")

    return df

# ---------- Execution ----------
# Uncomment when you have the real file
# df_raw = load_paysim("PS_20174392719_1491204439457_log.csv")

# Synthetic sample that mimics PaySim structure (for testing)
np.random.seed(42)
n = 30000
n_fraud = 40

types = np.random.choice(["CASH_IN", "CASH_OUT", "DEBIT", "PAYMENT", "TRANSFER"], n, p=[0.2, 0.35, 0.05, 0.25, 0.15])
df_raw = pd.DataFrame({
    "step": np.sort(np.random.randint(1, 745, n)),
    "type": types,
    "amount": np.random.lognormal(8, 1.5, n),
    "nameOrig": [f"C{np.random.randint(1000000000, 9999999999)}" for _ in range(n)],
    "oldbalanceOrg": np.random.exponential(50000, n),
    "newbalanceOrig": np.random.exponential(50000, n),
    "nameDest": [f"C{np.random.randint(1000000000, 9999999999)}" if np.random.rand() > 0.3 else f"M{np.random.randint(1000000000, 9999999999)}" for _ in range(n)],
    "oldbalanceDest": np.random.exponential(30000, n),
    "newbalanceDest": np.random.exponential(30000, n),
    "isFraud": np.zeros(n, dtype=int),
    "isFlaggedFraud": np.zeros(n, dtype=int)
})

# Inject some frauds (mostly TRANSFER + CASH_OUT pattern)
fraud_idx = np.random.choice(n, n_fraud, replace=False)
df_raw.loc[fraud_idx, "isFraud"] = 1
df_raw.loc[fraud_idx, "type"] = np.random.choice(["TRANSFER", "CASH_OUT"], n_fraud)
df_raw.loc[fraud_idx, "amount"] *= 3.5   # frauds tend to be larger

df_raw.to_csv("artifacts/01_paysim_raw_cleaned.csv", index=False)
print("Module 1 complete → artifacts/01_paysim_raw_cleaned.csv")
print(df_raw["isFraud"].value_counts())

Module 1 complete → artifacts/01_paysim_raw_cleaned.csv
isFraud
0    29960
1       40
Name: count, dtype: int64


Module 2: Feature Engineering & Encoding
Purpose: Create derived features, encode categorical type, scale numerics, prepare clean X and y.

In [9]:
# ============================================================
# MODULE 2: FEATURE ENGINEERING & ENCODING
# Purpose: Create balance-difference features, encode transaction type,
#          scale numeric features, save X and y.
# ============================================================

from sklearn.preprocessing import LabelEncoder, RobustScaler

def preprocess_paysim(df: pd.DataFrame):
    """
    Feature engineering tailored to PaySim.
    Returns:
        X          – feature DataFrame
        y          – binary labels
        feature_names
        scaler
        type_encoder
    """
    df = df.copy()

    # ----- Derived features (safe even if balances are noisy) -----
    df["errorBalanceOrig"] = df["newbalanceOrig"] + df["amount"] - df["oldbalanceOrg"]
    df["errorBalanceDest"] = df["oldbalanceDest"] + df["amount"] - df["newbalanceDest"]
    df["log_amount"] = np.log1p(df["amount"])
    df["hour_of_day"] = df["step"] % 24          # approximate hour
    df["day"] = df["step"] // 24                 # approximate day

    # Orig / Dest type (Customer vs Merchant)
    df["orig_is_customer"] = df["nameOrig"].str.startswith("C").astype(int)
    df["dest_is_merchant"] = df["nameDest"].str.startswith("M").astype(int)

    # Encode transaction type
    type_encoder = LabelEncoder()
    df["type_encoded"] = type_encoder.fit_transform(df["type"])

    # Final feature set (avoid leaking pure post-fraud balances if desired)
    feature_cols = [
        "step", "amount", "log_amount", "hour_of_day", "day",
        "type_encoded", "orig_is_customer", "dest_is_merchant",
        "oldbalanceOrg", "newbalanceOrig", "oldbalanceDest", "newbalanceDest",
        "errorBalanceOrig", "errorBalanceDest"
    ]

    X = df[feature_cols].copy()
    y = df["isFraud"].astype(int)

    # Robust scaling (handles outliers in amount and balances)
    scaler = RobustScaler()
    X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=feature_cols)

    return X_scaled, y, feature_cols, scaler, type_encoder

# ---------- Execution ----------
X, y, feature_names, scaler, type_encoder = preprocess_paysim(df_raw)

X.to_csv("artifacts/02_paysim_features.csv", index=False)
pd.Series(y, name="isFraud").to_csv("artifacts/02_paysim_labels.csv", index=False)
pd.DataFrame({"feature": feature_names}).to_csv("artifacts/02_paysim_feature_names.csv", index=False)

print("Module 2 complete")
print("Feature matrix shape:", X.shape)
print("Fraud rate:", y.mean())

Module 2 complete
Feature matrix shape: (30000, 14)
Fraud rate: 0.0013333333333333333


Module 3: Graph Proposal (Account Graph)
Purpose: Build a transaction graph where nodes are transactions and edges connect transactions that share the same nameOrig or nameDest. This is the core Graph Proposal component.

In [11]:
# ============================================================
# MODULE 3: GRAPH PROPOSAL (ACCOUNT GRAPH)
# Purpose: Propose edges between transactions that share
#          the same origin or destination account.
#          This captures money-mule / account-takeover patterns.
# ============================================================

import torch
from collections import defaultdict
import pandas as pd
import numpy as np

# Install torch_geometric and its dependencies if not already installed
try:
    from torch_geometric.data import Data
    import torch_geometric
except ImportError:
    print("torch_geometric not found or incompatible. Installing/Reinstalling...")
    # Get PyTorch version
    TORCH_VERSION = torch.__version__
    # Format it for PyG wheels (e.g., '2.1.0+cu118' -> '2.1.0')
    TORCH_VERSION_SIMPLE = TORCH_VERSION.split('+')[0]
    # Get CUDA version (e.g., '11.8' -> 'cu118')
    if torch.cuda.is_available():
        CUDA_VERSION = 'cu' + torch.version.cuda.replace('.', '')
    else:
        CUDA_VERSION = 'cpu'

    # Install torch-scatter and torch-sparse first, matching PyTorch and CUDA
    !pip install -q torch-scatter -f https://data.pyg.org/whl/torch-{TORCH_VERSION_SIMPLE}+{CUDA_VERSION}.html
    !pip install -q torch-sparse -f https://data.pyg.org/whl/torch-{TORCH_VERSION_SIMPLE}+{CUDA_VERSION}.html
    # Then install torch-geometric
    !pip install -q torch-geometric

    # Try importing again after installation
    from torch_geometric.data import Data
    import torch_geometric

def propose_account_graph(df: pd.DataFrame, X: pd.DataFrame, y: pd.Series,
                          max_edges_per_account: int = 50):
    """
    Graph Proposal strategy for PaySim:
    - Nodes = individual transactions
    - Edge between two transactions if they share nameOrig or nameDest
    - Limit degree to keep the graph manageable
    """
    n = len(df)
    edge_src, edge_dst = [], []

    # Build inverted index: account → list of transaction indices
    orig_groups = df.groupby("nameOrig").groups
    dest_groups = df.groupby("nameDest").groups

    def add_edges_from_groups(groups):
        for account, idxs in groups.items():
            idxs = list(idxs)
            if len(idxs) < 2:
                continue
            # Sort by time (step) so edges respect temporal order
            idxs_sorted = sorted(idxs, key=lambda i: df.loc[i, "step"])
            # Connect consecutive transactions of the same account (chain)
            # + a few random long-range connections
            for i in range(len(idxs_sorted) - 1):
                a, b = idxs_sorted[i], idxs_sorted[i + 1]
                edge_src.extend([a, b])
                edge_dst.extend([b, a])
            # Limit very high-degree accounts
            if len(idxs_sorted) > max_edges_per_account:
                continue

    add_edges_from_groups(orig_groups)
    add_edges_from_groups(dest_groups)

    edge_index = torch.tensor([edge_src, edge_dst], dtype=torch.long)

    x = torch.tensor(X.values, dtype=torch.float)
    y_tensor = torch.tensor(y.values, dtype=torch.long)

    data = Data(x=x, edge_index=edge_index, y=y_tensor)

    # Persist
    pd.DataFrame({"src": edge_src, "dst": edge_dst}).to_csv(
        "artifacts/03_paysim_proposed_edges.csv", index=False)
    torch.save(data, "artifacts/03_paysim_graph_data.pt")

    print(f"Proposed graph → {data.num_nodes} nodes, {data.num_edges} edges")
    return data

# ---------- Execution ----------
graph_data = propose_account_graph(df_raw, X, y, max_edges_per_account=40)
print("Module 3 complete")

torch_geometric not found or incompatible. Installing/Reinstalling...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 105.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 66.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 44.4 MB/s eta 0:00:00
Proposed graph → 30000 nodes, 0 edges
Module 3 complete


Module 4: Model Definition — SpaTempFraudNet-PaySim
Purpose: Hybrid architecture combining Graph Attention (spatial), Temporal encoding, and Adaptive Transformer.

In [12]:
# ============================================================
# MODULE 4: MODEL DEFINITION (SpaTempFraudNet-PaySim)
# Purpose: Adaptive Spatio-Temporal Graph Transformer
#          tailored for PaySim account graph + step time.
# ============================================================

import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GATConv

class SpaTempFraudNetPaySim(nn.Module):
    """
    SpaTempFraudNet-PaySim
    Components:
    1. Input projection of engineered features
    2. Temporal encoding from 'step'
    3. Graph Attention layers on the proposed account graph
    4. Lightweight Transformer encoder (adaptive attention)
    5. Binary classification head
    """
    def __init__(self, in_dim: int, hidden_dim: int = 64, n_heads: int = 4, dropout: float = 0.3):
        super().__init__()

        self.input_proj = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        # Temporal encoding (step)
        self.time_emb = nn.Linear(1, hidden_dim)

        # Spatial path – Graph Attention
        self.gat1 = GATConv(hidden_dim, hidden_dim // n_heads, heads=n_heads, dropout=dropout)
        self.gat2 = GATConv(hidden_dim, hidden_dim // n_heads, heads=n_heads, dropout=dropout)

        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)

        # Adaptive self-attention
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=n_heads,
            dim_feedforward=hidden_dim * 2,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=1)

        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, x, edge_index, time_feat=None):
        h = self.input_proj(x)

        if time_feat is not None:
            h = h + self.time_emb(time_feat)

        # Graph message passing with residuals
        h = h + F.elu(self.gat1(h, edge_index))
        h = self.norm1(h)
        h = h + F.elu(self.gat2(h, edge_index))
        h = self.norm2(h)

        # Adaptive Transformer
        h = h.unsqueeze(0)          # [1, N, D]
        h = self.transformer(h)
        h = h.squeeze(0)

        return self.classifier(h).squeeze(-1)

Module 5: Training with Focal Loss
Purpose: Train under extreme imbalance. Save best model and training history.

In [13]:
# ============================================================
# MODULE 5: TRAINING
# Purpose: Train SpaTempFraudNet-PaySim with Focal Loss.
# ============================================================

from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score, roc_auc_score

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.999, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        pt = torch.exp(-bce)
        return (self.alpha * (1 - pt) ** self.gamma * bce).mean()

def train_model(graph_data, X, y, epochs=30, hidden_dim=64):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Using device:", device)

    idx = np.arange(len(y))
    train_idx, val_idx = train_test_split(idx, test_size=0.2, stratify=y, random_state=42)

    model = SpaTempFraudNetPaySim(in_dim=X.shape[1], hidden_dim=hidden_dim).to(device)
    criterion = FocalLoss(alpha=0.999, gamma=2.0)
    optimizer = AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
    scheduler = CosineAnnealingLR(optimizer, T_max=epochs)

    x = graph_data.x.to(device)
    edge_index = graph_data.edge_index.to(device)
    y_tensor = graph_data.y.float().to(device)

    # Temporal feature from 'step'
    time_feat = torch.tensor(X["step"].values if "step" in X.columns else np.zeros(len(X)),
                             dtype=torch.float).unsqueeze(1).to(device)
    time_feat = (time_feat - time_feat.mean()) / (time_feat.std() + 1e-8)

    best_pr = 0.0
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        logits = model(x, edge_index, time_feat)
        loss = criterion(logits[train_idx], y_tensor[train_idx])
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(x, edge_index, time_feat)
            val_probs = torch.sigmoid(val_logits[val_idx]).cpu().numpy()
            pr = average_precision_score(y_tensor[val_idx].cpu().numpy(), val_probs)
            roc = roc_auc_score(y_tensor[val_idx].cpu().numpy(), val_probs)

        history.append({"epoch": epoch, "loss": loss.item(), "PR_AUC": pr, "ROC_AUC": roc})

        if pr > best_pr:
            best_pr = pr
            torch.save(model.state_dict(), "artifacts/05_paysim_best_model.pt")

        if epoch % 5 == 0 or epoch == 1:
            print(f"Epoch {epoch:02d} | Loss {loss.item():.4f} | Val PR-AUC {pr:.4f} | ROC-AUC {roc:.4f}")

    pd.DataFrame(history).to_csv("artifacts/05_paysim_training_history.csv", index=False)
    print(f"Best validation PR-AUC: {best_pr:.4f}")
    return model, train_idx, val_idx, device, time_feat

# ---------- Execution ----------
model, train_idx, val_idx, device, time_feat = train_model(graph_data, X, y, epochs=25)

Using device: cuda
Epoch 01 | Loss 0.2164 | Val PR-AUC 0.0022 | ROC-AUC 0.5102
Epoch 05 | Loss 0.1301 | Val PR-AUC 0.0022 | ROC-AUC 0.5055
Epoch 10 | Loss 0.0760 | Val PR-AUC 0.0021 | ROC-AUC 0.5015
Epoch 15 | Loss 0.0540 | Val PR-AUC 0.0021 | ROC-AUC 0.5022
Epoch 20 | Loss 0.0459 | Val PR-AUC 0.0022 | ROC-AUC 0.4982
Epoch 25 | Loss 0.0445 | Val PR-AUC 0.0022 | ROC-AUC 0.4965
Best validation PR-AUC: 0.0022


Module 6: Complete Fraud Evaluation Code
Purpose: Compute all required metrics at multiple operating thresholds and save predictions.

In [14]:
# ============================================================
# MODULE 6: COMPLETE EVALUATION
# Purpose: PR-AUC, ROC-AUC, Precision, Recall, F1, MCC
# ============================================================

from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    precision_score, recall_score, f1_score,
    matthews_corrcoef, confusion_matrix
)

def evaluate_fraud(y_true, y_prob, threshold=0.3, prefix="val"):
    y_pred = (y_prob >= threshold).astype(int)

    metrics = {
        "PR_AUC"    : average_precision_score(y_true, y_prob),
        "ROC_AUC"   : roc_auc_score(y_true, y_prob),
        "Precision" : precision_score(y_true, y_pred, zero_division=0),
        "Recall"    : recall_score(y_true, y_pred, zero_division=0),
        "F1"        : f1_score(y_true, y_pred, zero_division=0),
        "MCC"       : matthews_corrcoef(y_true, y_pred),
        "Threshold" : threshold
    }

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    metrics.update({"TN": tn, "FP": fp, "FN": fn, "TP": tp})

    pd.DataFrame([metrics]).to_csv(
        f"artifacts/06_paysim_{prefix}_metrics_th{threshold}.csv", index=False)

    pd.DataFrame({
        "y_true": y_true,
        "y_prob": y_prob,
        "y_pred": y_pred
    }).to_csv(f"artifacts/06_paysim_{prefix}_predictions.csv", index=False)

    print(f"\n===== {prefix.upper()} @ threshold = {threshold} =====")
    for k, v in metrics.items():
        print(f"{k:12s}: {v:.4f}" if isinstance(v, float) else f"{k:12s}: {v}")
    return metrics

# ---------- Execution ----------
model.load_state_dict(torch.load("artifacts/05_paysim_best_model.pt", map_location=device))
model.eval()

x = graph_data.x.to(device)
edge_index = graph_data.edge_index.to(device)

with torch.no_grad():
    logits = model(x, edge_index, time_feat)
    probs = torch.sigmoid(logits).cpu().numpy()

y_np = y.values
print("Probability stats → min / mean / max:",
      round(probs.min(), 4), round(probs.mean(), 4), round(probs.max(), 4))

for th in [0.5, 0.3, 0.2, 0.1, 0.05]:
    evaluate_fraud(y_np[val_idx], probs[val_idx], threshold=th, prefix="validation")

Probability stats → min / mean / max: 0.2937 0.4091 0.5144

===== VALIDATION @ threshold = 0.5 =====
PR_AUC      : 0.0022
ROC_AUC     : 0.5055
Precision   : 0.0000
Recall      : 0.0000
F1          : 0.0000
MCC         : -0.0066
Threshold   : 0.5000
TN          : 5802
FP          : 190
FN          : 8
TP          : 0

===== VALIDATION @ threshold = 0.3 =====
PR_AUC      : 0.0022
ROC_AUC     : 0.5055
Precision   : 0.0013
Recall      : 1.0000
F1          : 0.0027
MCC         : 0.0000
Threshold   : 0.3000
TN          : 0
FP          : 5992
FN          : 0
TP          : 8

===== VALIDATION @ threshold = 0.2 =====
PR_AUC      : 0.0022
ROC_AUC     : 0.5055
Precision   : 0.0013
Recall      : 1.0000
F1          : 0.0027
MCC         : 0.0000
Threshold   : 0.2000
TN          : 0
FP          : 5992
FN          : 0
TP          : 8

===== VALIDATION @ threshold = 0.1 =====
PR_AUC      : 0.0022
ROC_AUC     : 0.5055
Precision   : 0.0013
Recall      : 1.0000
F1          : 0.0027
MCC         : 0.0000
Th

Module 7: LLM-Based Trustworthy Explanations
Purpose: Select high-risk transactions and generate human-readable explanations that reference the graph structure and temporal context.

In [15]:
# ============================================================
# MODULE 7: LLM TRUSTWORTHY EXPLANATIONS
# Purpose: Generate transparent explanations for high-risk cases.
#          Ready to be replaced by a real LLM call.
# ============================================================

def generate_explanation(idx, prob, row):
    """
    Explanation template that mentions:
    - Transaction type and amount
    - Temporal context (step)
    - Account involvement (nameOrig / nameDest)
    - Graph-based relational signal
    """
    return (
        f"Transaction index {idx} flagged as FRAUD with probability {prob:.3f}.\n"
        f"Details:\n"
        f"  • Type          : {row.get('type', 'N/A')}\n"
        f"  • Amount        : {row.get('amount', 0):.2f}\n"
        f"  • Step (hour)   : {row.get('step', 'N/A')}\n"
        f"  • Origin account: {row.get('nameOrig', 'N/A')}\n"
        f"  • Dest account  : {row.get('nameDest', 'N/A')}\n\n"
        f"Reasoning: The Graph Proposal module connected this transaction "
        f"to other transactions sharing the same origin or destination account. "
        f"Combined with the Adaptive Transformer’s attention on amount, type, "
        f"and temporal features, the model detected a pattern consistent with "
        f"account-takeover or money-mule behavior (typical TRANSFER → CASH_OUT sequence)."
    )

risk_threshold = 0.15
high_risk_mask = probs >= risk_threshold
high_risk_idx = np.where(high_risk_mask)[0]
high_risk_prob = probs[high_risk_mask]

TOP_K = 15
if len(high_risk_idx) > TOP_K:
    order = np.argsort(high_risk_prob)[::-1][:TOP_K]
    high_risk_idx = high_risk_idx[order]
    high_risk_prob = high_risk_prob[order]

explanations = []
for idx, p in zip(high_risk_idx, high_risk_prob):
    row = df_raw.iloc[idx]
    explanations.append({
        "index": int(idx),
        "Fraud_Probability": round(float(p), 4),
        "type": row.get("type"),
        "amount": round(float(row.get("amount", 0)), 2),
        "step": int(row.get("step", 0)),
        "Explanation": generate_explanation(idx, p, row)
    })

exp_df = pd.DataFrame(explanations)
exp_df.to_csv("artifacts/07_paysim_llm_explanations.csv", index=False)

print(f"Saved {len(exp_df)} explanations → artifacts/07_paysim_llm_explanations.csv")
print(exp_df[["index", "Fraud_Probability", "type", "amount"]].head())

Saved 15 explanations → artifacts/07_paysim_llm_explanations.csv
   index  Fraud_Probability      type   amount
0  27328             0.5144   CASH_IN  3279.77
1  28673             0.5137  CASH_OUT  5713.87
2  29949             0.5127  CASH_OUT  7165.94
3  26625             0.5124   CASH_IN  6652.55
4  28944             0.5121  CASH_OUT  1088.90
